The [EKF section](/aiml-common/lectures/rse/ekf/index) pushed a Gaussian belief through a U-shaped function and found that the tangent line fails where the function is flat at the mean of the belief: the linearized variance drops to zero although the true output still spreads. This section shows the same failure on a robot. A TurtleBot drives straight ahead with its heading known only roughly, and one prediction step of the EKF is compared with samples pushed through the motion model itself.

The prediction uses the `EKF` class of the EKF section, built on filterpy's `ExtendedKalmanFilter`:


In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

plt.rcParams.update(
    {
        "figure.figsize": (6.5, 3.5),  # matches the 6.5 in text column
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.prop_cycle": cycler(
            color=[
                "#0072B2",
                "#D55E00",
                "#009E73",
                "#cb77a6",
                "#c28600",
                "#1d99e0",
                "#7f7f7f",
            ]
        ),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "svg.fonttype": "path",
    }
)

# named colors from the palette above
BLUE, VERMILLION, GREEN, PINK, GOLD, SKY, GRAY = (
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#cb77a6",
    "#c28600",
    "#1d99e0",
    "#7f7f7f",
)

np.random.seed(13)

In [ ]:
n_samples = 100_000  # samples for the reference prediction


from filterpy.kalman import ExtendedKalmanFilter


class EKF(ExtendedKalmanFilter):
    """Extended Kalman filter with a nonlinear motion model.

    Naming in the parent class: x is the mean (mu) and P the covariance (Sigma).
    Q is the motion noise and R the measurement noise, as above.
    """

    def __init__(self, dim_s, dim_z, g, G):
        # dim_s: size of the state; dim_z: size of a measurement
        super().__init__(dim_x=dim_s, dim_z=dim_z)
        self.g, self.G = g, G  # motion model g(a, s) and its Jacobian G(a, s)

    def predict(self, a):
        """Prediction step for the action a: lines 1 and 2 of the algorithm."""
        # Evaluate the Jacobian at the previous mean, before the mean moves.
        G_t = self.G(a, self.x)  # Jacobian at mu_{t-1}
        # Line 1: push the mean through the full nonlinear motion model.
        self.x = self.g(a, self.x)
        # Line 2: push the covariance through the linearized model, then add
        # the motion noise.
        self.P = G_t @ self.P @ G_t.T + self.Q
        # Keep a copy of the predicted belief, as the parent class does.
        self.x_prior, self.P_prior = self.x.copy(), self.P.copy()


# Lines 3 to 5 are the parent's update(z, HJacobian, Hx), where HJacobian(s)
# returns H_t and Hx(s) returns h(s), both evaluated at the predicted mean.

## The TurtleBot motion model

The straight drive at the start of the [EKF section](/aiml-common/lectures/rse/ekf/index#nonlinear-motion) is a special case of the differential-drive motion model of the TurtleBot. As in [nonholonomic robots](/aiml-common/lectures/kinematics/nonholonomic-robots/index), two wheels of radius $r$ sit at distance $d$ on either side of the chassis origin. The action is the pair of wheel speeds $a_t = (u_L, u_R)$, which set the forward speed and the turning rate, as [derived there](/aiml-common/lectures/kinematics/nonholonomic-robots/index#wheel-speeds-of-a-differential-drive) from rolling without slipping:

$$
v = \frac{r}{2}(u_L + u_R), \qquad \omega = \frac{r}{2d}(u_R - u_L).
$$

With the state $s = (x, y, \theta)$ and a time step $\Delta t$, the motion model and its Jacobian are

$$
g(a_t, s_{t-1}) = \begin{bmatrix} x + v\,\Delta t\cos\theta \\ y + v\,\Delta t\sin\theta \\ \theta + \omega\,\Delta t \end{bmatrix}, \qquad G_t = \begin{bmatrix} 1 & 0 & -v\,\Delta t\sin\theta \\ 0 & 1 & v\,\Delta t\cos\theta \\ 0 & 0 & 1 \end{bmatrix}.
$$

This model moves the robot $v\,\Delta t$ along its current heading and then turns it by $\omega\,\Delta t$. It is the first-order approximation of the motion model in [EKF localization](/aiml-common/lectures/rse/ekf-localization/index#motion-model), which holds the body twist constant over the step and moves the robot along the exact arc, $T_t = T_{t-1}\exp([\mathcal{V}_b]\,\Delta t)$. Expanding that arc in powers of $\Delta t$ gives

$$
\begin{aligned}
x_t &= x + v\,\Delta t\cos\theta - \tfrac{1}{2}\, v\,\omega\,\Delta t^2 \sin\theta + O(\Delta t^3), \\
y_t &= y + v\,\Delta t\sin\theta + \tfrac{1}{2}\, v\,\omega\,\Delta t^2 \cos\theta + O(\Delta t^3).
\end{aligned}
$$

The model here keeps the first-order terms and leaves out the $\Delta t^2$ terms, a sideways drift that the robot builds up by turning during the step. The two models agree exactly when $\omega = 0$. The drive below is straight, so its results are the same under either model.

The action $a_t = (u_L, u_R)$ is in $\mathbb R^2$ and the state is in $\mathbb R^3$, so $g$ maps $\mathbb R^3 \times \mathbb R^2$ to $\mathbb R^3$. The EKF holds the action at its given value and differentiates with respect to the state only. In the notation of the [Jacobian note](/aiml-common/lectures/rse/ekf/index#linearizing-the-function) of the EKF section, the map is $s_{t-1} \mapsto g(a_t, s_{t-1})$ from $\mathbb R^3$ to $\mathbb R^3$, so $n = m = 3$ and $G_t$ is $3 \times 3$. The derivative with respect to the two wheel speeds is a $3 \times 2$ matrix. The [EKF localization](/aiml-common/lectures/rse/ekf-localization/index) section uses it to turn wheel-speed noise into motion noise.

Let the robot start at the origin and drive straight down the page, $u_L = u_R$, with its heading known only roughly. The heading is uncertain even when every sensor works as specified: wheel slip and small errors in the wheel radii add heading error at every step of odometry, the [gyro's bias](/aiml-common/lectures/sensor-models/imu/index#noise-and-bias) integrates into a heading drift, and after startup, or after the robot is carried to a new spot, it has no heading reference until it sees a landmark. Write the heading as $\theta = -90° + \psi$, where $\psi$ is the heading error. The figure shows where the robot ends up.


In [ ]:
L = 1.0  # length of the drive, v * dt, in meters
psi_ex = np.deg2rad(40)  # one heading error, for the drawing
end = L * np.array([np.sin(psi_ex), -np.cos(psi_ex)])

fig, (ax_g, ax_f) = plt.subplots(2, 1, figsize=(6.5, 7.4), gridspec_kw={"height_ratios": [1.3, 1]})

# top: the geometry of one drive
arc = np.linspace(-np.pi / 3, np.pi / 3, 200)
ax_g.plot(L * np.sin(arc), -L * np.cos(arc), color=GRAY, ls=":", lw=1.2, label="end points of a 1 m drive")
ax_g.annotate("", xy=(0, -L), xytext=(0, 0), arrowprops={"arrowstyle": "-|>", "color": BLUE, "lw": 1.5})
ax_g.annotate("", xy=end, xytext=(0, 0), arrowprops={"arrowstyle": "-|>", "color": VERMILLION, "lw": 1.5})
ax_g.text(end[0] / 2 + 0.05, end[1] / 2 + 0.02, r"$v\,\Delta t$", color=VERMILLION, ha="left", va="bottom")  # hypotenuse
ax_g.text(-0.04, end[1] / 2, r"$v\,\Delta t\cos\psi$", color=BLUE, ha="right", va="center")  # adjacent side
q = 0.04  # right-angle mark where the adjacent side meets the opposite side
ax_g.plot([0, q, q], [end[1] + q, end[1] + q, end[1]], color="black", lw=0.8)
ax_g.plot([], [], color=BLUE, lw=1.5, label=r"intended drive, $\psi = 0$")
ax_g.plot([], [], color=VERMILLION, lw=1.5, label=r"actual drive, heading error $\psi$")
a = np.linspace(0, psi_ex, 30)
ax_g.plot(0.3 * np.sin(a), -0.3 * np.cos(a), color="black", lw=0.8)
ax_g.text(0.08, -0.4, r"$\psi$", fontsize=10)
ax_g.plot([0, end[0]], [end[1], end[1]], color=GREEN, lw=2)
ax_g.text(end[0] / 2, end[1] + 0.03, r"$v\,\Delta t \sin\psi$", ha="center", va="bottom", color=GREEN)
ax_g.plot([0, end[0]], [-L, -L], color=GRAY, ls="--", lw=0.8)  # the intended forward distance
ax_g.plot([end[0], end[0]], [end[1], -L], color=PINK, lw=4, solid_capstyle="butt")
ax_g.text(end[0] - 0.04, (end[1] - L) / 2, r"$v\,\Delta t\,(1 - \cos\psi)$", ha="right", va="center", color=PINK)
ax_g.plot(0, 0, "s", color="black", ms=6)
ax_g.text(0.04, 0.03, "start", fontsize=8)
ax_g.set_xlim(-1.0, 1.0)
ax_g.set_ylim(-1.1, 0.1)
ax_g.set_aspect("equal")
ax_g.set_xlabel("$x$ (m), sideways")
ax_g.set_ylabel("$y$ (m), forward is down")
ax_g.set_title("One 1 m drive with a heading error")
ax_g.legend(loc="upper left", fontsize=7.5)

# bottom: the two pieces as functions of the heading error
psi = np.linspace(-np.pi / 3, np.pi / 3, 400)
deg = np.rad2deg(psi)
ax_f.plot(deg, L * np.sin(psi), color=GREEN, label=r"sideways $v\,\Delta t\sin\psi$")
ax_f.plot(deg, L * psi, color=GREEN, ls="--", lw=1, label=r"first order, $v\,\Delta t\,\psi$")
ax_f.plot(deg, L * (1 - np.cos(psi)), color=PINK, label=r"shortfall $v\,\Delta t\,(1 - \cos\psi)$")
ax_f.plot(deg, L * psi**2 / 2, color=PINK, ls="--", lw=1, label=r"second order, $v\,\Delta t\,\psi^2/2$")
ax_f.axvspan(-30, 30, color=BLUE, alpha=0.08, lw=0, label=r"$\pm 30°$")
ax_f.set_xlim(-60, 60)
ax_f.set_xticks(range(-60, 61, 30))
ax_f.set_xlabel(r"heading error $\psi$ (degrees)")
ax_f.set_ylabel("meters, for a 1 m drive")
ax_f.legend(loc="upper left", fontsize=7.5, ncol=2)
ax_f.set_ylim(-1.0, 1.15)
fig.tight_layout()
plt.show()

The heading is $\theta = -90° + \psi$, so the drive of length $v\,\Delta t$ points $\psi$ away from straight down. The drive is the hypotenuse of a right triangle, and its two legs are the sideways offset and the forward progress. Its components are $x = v\,\Delta t\cos\theta$ and $y = v\,\Delta t\sin\theta$. With $\cos(-90° + \psi) = \sin\psi$ and $\sin(-90° + \psi) = -\cos\psi$ they become

$$
x = v\,\Delta t\sin\psi, \qquad y = -v\,\Delta t\cos\psi .
$$

Whatever the heading error, the end point lies on the dotted circle of radius $v\,\Delta t$ around the start. For small $\psi$, $\sin\psi \approx \psi$ and $\cos\psi \approx 1 - \psi^2/2$, so

$$
x \approx v\,\Delta t\,\psi, \qquad y \approx -v\,\Delta t + \frac{v\,\Delta t}{2}\,\psi^2 .
$$

The sideways offset (green) is first order in the heading error. It grows in proportion to $\psi$, and in the lower panel it stays on its straight line across the $\pm 30°$ band. The shortfall in forward progress (pink) is second order. It is flat at $\psi = 0$ and grows like $\psi^2$, and it is the same for $+\psi$ and $-\psi$, so the robot always stops short of the intended 1 m and never goes past it. A small heading error moves the robot sideways much more than it shortens the drive: at $\psi = 10°$ the offset is 17 cm and the shortfall 1.5 cm.

## Sampled and EKF predictions

The code below uses the geometry of a TurtleBot3 Waffle and drives it straight down the page at 0.25 m/s for 4 s, starting from a belief that knows the position to 2 cm but the heading only to $30°$. It predicts the new pose twice: by pushing samples of the starting pose through $g$, and with one prediction step of the `EKF` class above. To isolate the effect of the heading, the motion noise $Q_t$ is left out.

Before looking at the result, evaluate $G_t$ at $\theta = -90°$ and decide how much of the heading uncertainty the EKF passes into the sideways coordinate $x$ and how much into the forward coordinate $y$.

In [ ]:
# Robot geometry, in meters: TurtleBot3 Waffle wheel radius r and half the
# distance between the wheels d.
r, d = 0.033, 0.1435
dt = 4.0  # length of the time step, in seconds


def g_dd(a, s):
    """Differential-drive motion model. s has shape (3, n): one pose (x, y, theta) per column."""
    u_L, u_R = a  # wheel speeds, in rad/s
    # forward speed v and turning rate w from the wheel speeds
    v, w = r / 2 * (u_L + u_R), r / (2 * d) * (u_R - u_L)
    x, y, th = s
    # Move forward v * dt along the current heading, then turn by w * dt.
    return np.vstack([x + v * dt * np.cos(th), y + v * dt * np.sin(th), th + w * dt])


def G_dd(a, s):
    """Jacobian of g_dd with respect to the state, at the pose s of shape (3, 1)."""
    u_L, u_R = a
    v, th = r / 2 * (u_L + u_R), s[2, 0]
    # Only the heading enters nonlinearly, so only the third column differs
    # from the identity: it says how x and y change when theta changes.
    return np.array([[1, 0, -v * dt * np.sin(th)], [0, 1, v * dt * np.cos(th)], [0, 0, 1]])


# The action: both wheels at the same speed, so the robot drives straight
# at 0.25 m/s. Over dt = 4 s it covers 1 m.
u = 0.25 / r  # wheel speed in rad/s: with u_L = u_R = u, v = r u, so u = v / r
a = (u, u)

# The starting belief: the robot is at the origin, facing straight down the
# page (theta = -90 degrees). Its position is known to 2 cm, its heading only
# to 30 degrees.
mu0 = np.array([[0.0], [0.0], [-np.pi / 2]])
# Sigma0 is 3 x 3, in the order of the state (x, y, theta). Its diagonal holds
# the variances, the squared standard deviations: (2 cm)^2 for x and y, in m^2,
# and (30 degrees)^2 for theta, in rad^2. The off-diagonal entries are the
# covariances between pairs of coordinates; zero says that an error in one
# coordinate tells nothing about the others.
Sigma0 = np.diag([0.02**2, 0.02**2, np.deg2rad(30) ** 2])

# Reference answer: draw starting poses from the belief and push each one
# through the motion model. Each column of s_samples is one predicted pose.
s_samples = g_dd(a, np.random.multivariate_normal(mu0.ravel(), Sigma0, n_samples).T)

# The EKF's answer: one prediction step from the same belief. The motion
# noise Q is zero so that only the heading uncertainty spreads the belief.
ekf = EKF(dim_s=3, dim_z=1, g=g_dd, G=G_dd)
# filterpy's names: x is the mean mu, P the covariance Sigma, Q the motion noise.
ekf.x, ekf.P, ekf.Q = mu0.copy(), Sigma0.copy(), np.zeros((3, 3))
ekf.predict(a)

# Compare the two predictions of the position.
print("sampled mean x, y:", s_samples[:2].mean(axis=1).round(3), "std x, y:", s_samples[:2].std(axis=1).round(3))
print("EKF     mean x, y:", ekf.x[:2, 0].round(3), "std x, y:", np.sqrt(np.diag(ekf.P)[:2]).round(3))

In [ ]:
# forward position after the drive as a function of the heading error psi,
# where theta = -90 degrees + psi: y(psi) = -v dt cos(psi)
vdt = r / 2 * (2 * u) * dt  # distance driven, 1 m
psi_sd = np.sqrt(Sigma0[2, 2])  # heading standard deviation, 30 degrees
psis = np.linspace(-np.pi / 2, np.pi / 2, 400)

fig, (ax_u, ax_y) = plt.subplots(1, 2, figsize=(6.5, 3.4))

ax_u.plot(np.rad2deg(psis), -vdt * np.cos(psis), color=GREEN, label=r"$y(\psi)$")
ax_u.axhline(-vdt, color=PINK, ls="--", label=r"tangent at $\psi = 0$")
ax_u.plot(0, -vdt, "o", color=PINK)
ax_u.axvspan(-2 * np.rad2deg(psi_sd), 2 * np.rad2deg(psi_sd), color=BLUE, alpha=0.12,
             label=r"heading belief $\pm 2\sigma$")
ax_u.set_xlim(-90, 90)
ax_u.set_ylim(-1.1, 0.3)
ax_u.set_xlabel(r"heading error $\psi$ (degrees)")
ax_u.set_ylabel("$y$ after the drive (m)")
ax_u.set_title("motion model along $y$")
ax_u.legend(loc="upper center")

ys_m = np.linspace(-1.1, 0.1, 600)
sy = np.sqrt(ekf.P[1, 1])
ax_y.hist(s_samples[1], bins=np.linspace(-1.1, 0.1, 121), density=True, color=VERMILLION, alpha=0.35,
          label="samples")
ax_y.plot(ys_m, np.exp(-((ys_m - ekf.x[1, 0]) ** 2) / (2 * sy**2)) / np.sqrt(2 * np.pi * sy**2),
          color=PINK, ls="--", label="EKF")
ax_y.axvline(s_samples[1].mean(), color=VERMILLION, ls=":", label="sampled mean")
ax_y.set_ylim(0, 12)
ax_y.set_xlabel("$y$ (m)")
ax_y.set_title("forward position $y$")
ax_y.legend(loc="upper right")

fig.tight_layout()
plt.show()

The left panel is the motion model along $y$, drawn the same way as the U-shaped function $g$ in the [EKF section](/aiml-common/lectures/rse/ekf/index#a-gaussian-through-a-u-shaped-function). The curve is the forward position after the drive as a function of the heading error, $y(\psi) = -v\,\Delta t\cos\psi$, and the shaded band is the heading belief out to two standard deviations, $\pm 60°$. It is the U again, with its lowest point at the mean of the belief, $\psi = 0$. The EKF replaces the curve with its tangent line at the mean. That line is flat, because the Jacobian entry $\partial y / \partial\theta = v\,\Delta t\cos\theta$ is zero at $\theta = -90°$. This is the case of the belief centered at the bottom of the U in [When linearization fails](/aiml-common/lectures/rse/ekf/index#when-linearization-fails), where the linearized variance was zero.

The right panel shows the result. The sampled forward position stops at $-1$ m and spreads toward shorter progress, and its mean is $-0.87$ m. The EKF puts all of its probability in a narrow spike at $-1$ m, only as wide as the 2 cm starting uncertainty. It reports almost no uncertainty about how far the robot went, and it is wrong.

The same samples, drawn in the plane, show where the robot can end up.

In [ ]:
def ellipse(mean, cov, k=2.0, n=200):
    """Points on the k-sigma ellipse of a 2D Gaussian."""
    vals, vecs = np.linalg.eigh(cov)
    t = np.linspace(0, 2 * np.pi, n)
    return mean[:, None] + vecs @ (k * np.sqrt(vals)[:, None] * np.vstack([np.cos(t), np.sin(t)]))


fig, ax_xy = plt.subplots(figsize=(6.5, 3.6))

ax_xy.scatter(s_samples[0, :3000], s_samples[1, :3000], s=2, color=VERMILLION, alpha=0.3,
              rasterized=True, label="samples")
ell = ellipse(ekf.x[:2, 0], ekf.P[:2, :2])
ax_xy.plot(ell[0], ell[1], color=PINK, ls="--", label=r"EKF $2\sigma$")
ax_xy.plot(*s_samples[:2].mean(axis=1), "X", color=BLUE, ms=8, mec="white", label="sampled mean")
ax_xy.plot(*ekf.x[:2, 0], "o", color=PINK, ms=7, mec="white", label="EKF mean")
# the starting pose: at the origin, heading straight down the page
ax_xy.plot(0, 0, "s", color=GRAY, ms=7, mec="white", label="start")
ax_xy.annotate("", xy=(0, -0.2), xytext=(0, 0),
               arrowprops={"arrowstyle": "->", "color": GRAY, "lw": 1.5})
ax_xy.set_aspect("equal")
ax_xy.set_xlim(-1.2, 1.2)
ax_xy.set_ylim(-1.2, 0.3)
ax_xy.set_xlabel("$x$ (m)")
ax_xy.set_ylabel("$y$ (m)")
ax_xy.set_title("predicted position")
ax_xy.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

fig.tight_layout()
plt.show()

The gray square marks the start at the origin, and the arrow shows the heading straight down the page. The dots are 3,000 of the sampled positions. They lie on an arc. Every sample gets the same command, both wheels at the same speed, so it drives straight for exactly 1 m and never turns. The samples differ only in where they point at the start: the belief knows the starting heading only to 30°, and each sample draws one heading from it and keeps it for the whole drive. A 1 m straight drive along each of those headings ends on a circle of radius 1 m around the start, and the 2 cm uncertainty in the starting position gives the arc its thickness. The dashed ellipse is the EKF's prediction, drawn two standard deviations out from its mean. The prediction is one step from the starting belief $(\mu_0, \Sigma_0)$ to $(\bar\mu_1, \bar\Sigma_1)$. With $\bar\mu_1$ and $\bar\Sigma_1$ restricted to their $x, y$ parts, the ellipse is the set of positions $p$ with $\sqrt{(p - \bar\mu_1)^\top \bar\Sigma_1^{-1} (p - \bar\mu_1)} = 2$, and it holds about 86% of the probability of the EKF's Gaussian.

The shape of the ellipse comes from the Jacobian. With the motion noise left out, the EKF's predicted covariance is $\bar\Sigma_1 = G_1 \Sigma_0 G_1^\top$, with $G_1$ evaluated at the starting mean $\mu_0$. Only the third column of $G_1$, the one for $\theta$, differs from the identity. At the starting heading $\theta = -90°$ that column is $(v\,\Delta t, 0, 1)^\top$, because $-v\,\Delta t\sin\theta = v\,\Delta t$ and $v\,\Delta t\cos\theta = 0$. All of the heading variance therefore goes into $x$ and none into $y$:

$$
(\bar\Sigma_1)_{xx} = \sigma_x^2 + (v\,\Delta t)^2\sigma_\theta^2, \qquad (\bar\Sigma_1)_{yy} = \sigma_y^2, \qquad (\bar\Sigma_1)_{xy} = 0.
$$

With $v\,\Delta t = 1$ m, $\sigma_\theta = 30° = 0.52$ rad and $\sigma_x = \sigma_y = 2$ cm, the EKF's standard deviation is 0.52 m in $x$ and 0.02 m in $y$. The cross term is zero, so the ellipse lines up with the axes.

In $x$ the EKF is close to the samples. Sideways, the robot ends at $x = v\,\Delta t\sin\psi$, and $\sin\psi \approx \psi$ for small heading errors, so the tangent line stays close to the curve. The samples spread 0.46 m in $x$, a little less than the EKF's 0.52 m, because $\sin\psi$ falls below $\psi$ for the larger heading errors in the tails of the belief. In $y$ the EKF is far off. The samples spread 0.17 m, more than eight times the EKF's 2 cm, because the tangent at the bottom of the U is flat and passes none of the heading uncertainty into the forward direction.

The samples show what actually happens. Each one is a starting pose that the belief allows, pushed through the motion model itself, so together they are the robot's distribution at $t = 1$: somewhere on the arc, on average 0.87 m forward, and about 1 m at most. The EKF describes the same moment with a Gaussian centered at the full meter that claims to know the forward position to 2 cm. That belief is overconfident twice over. Its mean puts the robot 13 cm farther forward than the samples' mean, and its spread in $y$ is less than an eighth of the real one. Nothing inside the filter shows the problem: $\bar\Sigma_1$ is small, so the belief looks trustworthy, and the next measurement update gives the measurement less weight than it deserves. This is the start of the filter divergence described in the EKF section under [When linearization fails](/aiml-common/lectures/rse/ekf/index#when-linearization-fails).

The [localization section](/aiml-common/lectures/rse/ekf-localization/index) continues this example: the robot measures the range and bearing to a landmark, and the EKF uses the measurement to correct its belief.